<a href="https://colab.research.google.com/github/valerio-unifei/ECAA08-2026.2-Projeto/blob/main/etapa-01-logica/04%20-%20Logica%20Proposicional%20Conectivos%20e%20Permissivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos

Neste notebook implementamos as funções de avaliação lógica proposicional baseadas na nossa aula teórica. Construiremos os blocos de permissivos de partida (*Start Permissives*) para a **Esteira Principal (M-101)** e para o **Alimentador do Silo (XV-101)**, além de estruturar o intertravamento contínuo (Trip).

In [ ]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

## 2.1. Permissivo da Esteira Principal (M-101)

O acionamento requer:
* Ausência de emergência ($e_1$)
* Ausência de alarme ($a_1$)
* Caixas com espaço ($batch\_full$)
* Modo de operação definido de forma exclusiva ($Auto \oplus Manual$)

In [ ]:
def permissivo_esteira_M101(e1: bool, a1: bool, batch_full: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    # Modo exclusivo (Auto XOR Manual)
    modo_valido = XOR(auto_mode, manual_mode)

    # Condição combinada de permissivo: ¬e1 ∧ ¬a1 ∧ ¬batch_full ∧ (Auto ⊕ Manual)
    permissivo = (NOT(e1) and NOT(a1) and NOT(batch_full) and modo_valido)

    # Trip imediato: e1 ∨ a1 ∨ batch_full
    trip = OR(OR(e1, a1), batch_full)

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Cenários de Teste
cenarios_m101 = [
    {"cenario": "Operação Normal (Auto)", "args": (False, False, False, True, False)},
    {"cenario": "Parada de Emergência", "args": (True, False, False, True, False)},
    {"cenario": "Alarme Ativo", "args": (False, True, False, True, False)},
    {"cenario": "Caixa de Loteamento Cheia", "args": (False, False, True, True, False)},
    {"cenario": "Conflito de Modo (Auto + Manual)", "args": (False, False, False, True, True)},
]

resultados = []
for c in cenarios_m101:
    res = permissivo_esteira_M101(*c["args"])
    resultados.append({
        "Cenário": c["cenario"],
        "Permissivo": res["Permissivo_Habilitado"],
        "Trip Ativo": res["Trip_Ativo"],
        "Modo Válido": res["Modo_Valido"]
    })

df_m101 = pd.DataFrame(resultados)
display(df_m101)

## 2.2. Permissivo de Alimentação do Silo (XV-101)

Libera a peça se a esteira já estiver rodando ($m_1$), o silo não estiver vazio ($\neg l_{silo}$), não houver peça presa na saída ($\neg s_{feed}$) e o permissivo da esteira M-101 ($P_{M-101}$) for verdadeiro.

In [ ]:
def permissivo_alimentador_XV101(m1: bool, l_silo: bool, s_feed: bool, p_m101: bool) -> bool:
    # Condição: m1 ∧ ¬l_silo ∧ ¬s_feed ∧ P_M-101
    return AND(AND(m1, NOT(l_silo)), AND(NOT(s_feed), p_m101))

# Cenários de Teste
cenarios_xv101 = [
    {"cenario": "Tudo OK - Pronto para Alimentar", "args": (True, False, False, True)},
    {"cenario": "Falha: Esteira M-101 Desligada", "args": (False, False, False, True)},
    {"cenario": "Falha: Silo Vazio", "args": (True, True, False, True)},
    {"cenario": "Falha: Peça Presa na Saída", "args": (True, False, True, True)},
    {"cenario": "Falha: Permissivo M-101 Falso", "args": (True, False, False, False)},
]

resultados_xv = []
for c in cenarios_xv101:
    res = permissivo_alimentador_XV101(*c["args"])
    resultados_xv.append({
        "Cenário": c["cenario"],
        "Permissivo XV-101 Habilitado": res
    })

df_xv101 = pd.DataFrame(resultados_xv)
display(df_xv101)

## Geração Automática de Tabela-Verdade para Validação Exaustiva (M-101)

Abaixo geramos todas as combinações possíveis para as condições de falha operacionais.

In [ ]:
variaveis_m101 = ['e1 (Emerg)', 'a1 (Alarme)', 'batch_full']
tabela_m101 = []

# Testando apenas as variaveis de falha, fixando modo Auto=True e Manual=False
for combo in itertools.product([False, True], repeat=len(variaveis_m101)):
    st = dict(zip(variaveis_m101, combo))
    res = permissivo_esteira_M101(st['e1 (Emerg)'], st['a1 (Alarme)'], st['batch_full'], True, False)
    
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela_m101.append(row)

df_tv_m101 = pd.DataFrame(tabela_m101)
print(f"Total de combinações de falha avaliadas: {len(df_tv_m101)}")
print(f"Combinações seguras que liberam a esteira: {df_tv_m101['Permissivo'].sum()}")
display(df_tv_m101)
